# Post 2 — TabPFN Experiments

Runs TabPFN CV experiments for the insurance pricing task.

**Requires:**
- GPU (RTX 4090 or better recommended)
- PriorLabs auth: set `TABPFN_TOKEN` env var (from https://ux.priorlabs.ai, accept license there)
- Baselines already run: `results/cv/` must contain GLM/GBM fold_metrics JSON files

Results saved to `results/cv/` — picked up by `post2_analysis.ipynb`.

**Experiments:**
1. TabPFN 10K — rate strategy — raw + engineered + GBM features
2. TabPFN 50K — rate strategy — best feature set from 10K run
3. Hurdle model — TabPFNClassifier for claim occurrence

In [ ]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Auth must be set before importing tabpfn
# Set TABPFN_TOKEN in env before launching Jupyter, or set it here:
# os.environ['TABPFN_TOKEN'] = 'your_token_here'
os.environ.setdefault('TABPFN_NO_BROWSER', 'true')

import numpy as np
import pandas as pd
import yaml
import warnings
warnings.filterwarnings('ignore')

from src.data.load_insurance import load_processed, get_dev_holdout
from src.features.insurance_features import RawFeaturePipeline, EngineeredFeaturePipeline, GBMFeaturePipeline
from src.models.glm import GammaGLM
from src.models.tabpfn_wrapper import TabPFNWrapper
from src.evaluation.cv_engine import run_cv, run_cv_hurdle

with open(os.path.join(project_root, 'configs/experiment_config.yaml')) as f:
    CONFIG = yaml.safe_load(f)

print('Imports OK')

## Step 1 — GPU + Auth Verification

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. TabPFN at 10K+ will be very slow on CPU.")
    print("Set TABPFN_ALLOW_CPU_LARGE_DATASET=1 to proceed anyway.")

token = os.environ.get('TABPFN_TOKEN', '')
print(f"\nTABPFN_TOKEN set: {bool(token)} (prefix: {token[:8]}...)" if token else "\nWARNING: TABPFN_TOKEN not set")

In [ ]:
# Minimal fit+predict smoke test — verifies auth + GPU in ~30s
import time
from tabpfn import TabPFNRegressor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
rng = np.random.default_rng(42)
X_tiny = rng.standard_normal((200, 9)).astype(np.float32)
y_tiny = np.abs(rng.standard_normal(200)).astype(np.float32)
X_val_tiny = rng.standard_normal((500, 9)).astype(np.float32)

t0 = time.time()
r = TabPFNRegressor(device=device)
r.fit(X_tiny, y_tiny)
preds = r.predict(X_val_tiny)
elapsed = time.time() - t0

print(f"TabPFN smoke test: fit+predict(500) in {elapsed:.1f}s on {device}")
print(f"Predictions: min={preds.min():.3f}, max={preds.max():.3f}, mean={preds.mean():.3f}")
print("Auth + GPU: OK" if elapsed < 120 else f"WARNING: slow ({elapsed:.0f}s) — check GPU")

TABPFN_AVAILABLE = True

## Step 2 — Data Loading (same splits as baselines)

In [ ]:
df = load_processed()
splits = get_dev_holdout(df)

X_dev, X_holdout     = splits['X_dev'], splits['X_holdout']
y_dev, y_holdout     = splits['y_dev'], splits['y_holdout']
exp_dev, exp_holdout = splits['exposure_dev'], splits['exposure_holdout']
cv_folds             = splits['cv_folds']

print(f"Dev: {len(X_dev):,} | Holdout: {len(X_holdout):,}")
print(f"CV folds loaded from disk — same as baselines run")

## Step 3 — TabPFN 10K: All Feature Sets

Rate strategy: target = pure_premium / Exposure, multiply back at inference.
Option B recalibration: 3 inner folds per outer fold.

Estimated time per fold on RTX 4090: ~3-5 min (3 inner fits + 1 outer fit + batched predict).
Total: ~75-125 min for all 3 feature sets × 5 folds.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tabpfn_cv = {}

for pipe_cls, pipe_label in [
    (RawFeaturePipeline, 'raw'),
    (EngineeredFeaturePipeline, 'engineered'),
    (GBMFeaturePipeline, 'gbm'),
]:
    tabpfn_cv[f'TabPFN_10K_{pipe_label}'] = run_cv(
        model_factory=lambda: TabPFNWrapper(
            task='regression',
            n_train_max=10_000,
            exposure_strategy='feature',
            predict_batch_size=5_000,
        ),
        feature_pipeline_factory=pipe_cls,
        X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
        cv_folds=cv_folds, approach='tweedie',
        model_name='TabPFN_10K', features_label=pipe_label,
        tabpfn_n_inner_folds=CONFIG['tabpfn']['n_inner_folds_recal'],
    )
    print(f"TabPFN_10K_{pipe_label} mean:",
          {k: round(v,4) for k,v in tabpfn_cv[f'TabPFN_10K_{pipe_label}']['mean_metrics'].items()})

## Step 4 — TabPFN 50K: Best Feature Set

Run only the best-performing feature set from Step 3 at 50K samples.
This is the fairest comparison against GBMs which train on ~430K rows.

In [ ]:
# Identify best feature set from 10K runs
best_pipe_label = min(
    ['raw', 'engineered', 'gbm'],
    key=lambda p: tabpfn_cv[f'TabPFN_10K_{p}']['mean_metrics']['tweedie_dev_1.5']
)
best_pipe_cls = {'raw': RawFeaturePipeline, 'engineered': EngineeredFeaturePipeline,
                 'gbm': GBMFeaturePipeline}[best_pipe_label]
print(f"Best 10K feature set: {best_pipe_label} — running 50K with this pipeline")

tabpfn_cv['TabPFN_50K_best'] = run_cv(
    model_factory=lambda: TabPFNWrapper(
        task='regression',
        n_train_max=50_000,
        exposure_strategy='feature',
        predict_batch_size=5_000,
    ),
    feature_pipeline_factory=best_pipe_cls,
    X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
    cv_folds=cv_folds, approach='tweedie',
    model_name='TabPFN_50K', features_label=best_pipe_label,
    tabpfn_n_inner_folds=CONFIG['tabpfn']['n_inner_folds_recal'],
)
print("TabPFN_50K mean:",
      {k: round(v,4) for k,v in tabpfn_cv['TabPFN_50K_best']['mean_metrics'].items()})

## Step 5 — Hurdle Model: TabPFN + GammaGLM

This is the experiment TabPFN is genuinely suited for.

**Why this is TabPFN's home ground:**
- Stage 1 (classification): binary claim occurrence, ~10K subsample — classic TabPFN task
- Stage 2 (severity regression): claims-only subset ~15-20K rows — within TabPFN's context window,
  no exposure complexity, pure pattern recognition on who has expensive claims

Two hurdle variants:
1. **TabPFN × TabPFN**: both stages use TabPFN (best-case scenario)
2. **TabPFN × GammaGLM**: TabPFN classifier + classical severity model (ablation)

Combined prediction: `P(claim > 0) × E[pure_premium | claim > 0]`

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Variant 1: TabPFN classifier (stage 1) + GammaGLM (stage 2)
# GammaGLM on ~15-20K claims rows is fast and well-calibrated.
# This isolates whether TabPFN's classification improves over logistic regression.
print("=== Hurdle: TabPFN cls + GammaGLM sev ===")
tabpfn_cv['TabPFN_hurdle_cls_gamma'] = run_cv_hurdle(
    stage1_factory=lambda: TabPFNWrapper(
        task='classification',
        n_train_max=10_000,
        predict_batch_size=5_000,
    ),
    stage2_factory=GammaGLM,
    feature_pipeline_factory=GBMFeaturePipeline,
    X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
    cv_folds=cv_folds,
    model_name='TabPFN_hurdle_cls_gamma', features_label='gbm',
    tabpfn_n_inner_folds=CONFIG['tabpfn']['n_inner_folds_recal'],
)
print("TabPFN cls + Gamma mean:",
      {k: round(v,4) for k,v in tabpfn_cv['TabPFN_hurdle_cls_gamma']['mean_metrics'].items()})

# Variant 2: TabPFN classifier (stage 1) + TabPFN regressor (stage 2)
# Stage 2 runs on claims-only: ~15-20K rows — well within TabPFN's 10K budget
# after subsampling. This is the "pure TabPFN" hurdle.
print("\n=== Hurdle: TabPFN cls + TabPFN sev ===")
tabpfn_cv['TabPFN_hurdle_full'] = run_cv_hurdle(
    stage1_factory=lambda: TabPFNWrapper(
        task='classification',
        n_train_max=10_000,
        predict_batch_size=5_000,
    ),
    stage2_factory=lambda: TabPFNWrapper(
        task='regression',
        n_train_max=10_000,
        exposure_strategy='feature',   # pure_premium is already rate; pass exposure as context
        predict_batch_size=5_000,
    ),
    feature_pipeline_factory=GBMFeaturePipeline,
    X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
    cv_folds=cv_folds,
    model_name='TabPFN_hurdle_full', features_label='gbm',
    tabpfn_n_inner_folds=CONFIG['tabpfn']['n_inner_folds_recal'],
)
print("TabPFN full hurdle mean:",
      {k: round(v,4) for k,v in tabpfn_cv['TabPFN_hurdle_full']['mean_metrics'].items()})

## Step 6 — Summary of All CV Results

In [ ]:
print("=== TabPFN CV Summary (Tweedie deviance, lower is better) ===")
summary = pd.DataFrame(
    {k: v['mean_metrics'] for k, v in tabpfn_cv.items()}
).T.round(4)
print(summary)

print("\n--- Key findings ---")
# Tweedie: best TabPFN config
tw_scores = {k: v['mean_metrics']['tweedie_dev_1.5'] for k, v in tabpfn_cv.items()}
best_key = min(tw_scores, key=tw_scores.get)
print(f"Best TabPFN Tweedie deviance: {best_key} = {tw_scores[best_key]:.4f}")
print(f"  vs LightGBM_gbm (from baselines): 79.55")
print(f"  vs GLM_engineered (from baselines): 183.37")

print("\n--- Gini comparison (hurdle vs direct Tweedie) ---")
for k, v in tabpfn_cv.items():
    g = v['mean_metrics'].get('gini', float('nan'))
    print(f"  {k}: gini={g:.4f}")

In [ ]:
## Step 7 — Holdout Evaluation + Merge for Auction

Refits all TabPFN configs on full dev set, evaluates on holdout.
Also fits hurdle models for holdout predictions.
Merges with baseline predictions into a single `predictions_all.parquet`
that `run_all_auctions` can read.

In [ ]:
import importlib
import src.evaluation.cv_engine as _cv_engine_mod
importlib.reload(_cv_engine_mod)
from src.evaluation.cv_engine import holdout_evaluation, holdout_evaluation_hurdle

from pathlib import Path

holdout_dir = Path(project_root) / 'results' / 'holdout'
holdout_dir.mkdir(parents=True, exist_ok=True)

all_preds = {}
all_metrics = []

# --- TabPFN Tweedie holdout ---
# Use exposure_strategy='feature': pure_premium is the direct target,
# exposure is passed as an input feature. This avoids the rate-strategy bug
# where pure_premium (already ClaimAmount/Exposure) would be divided by
# exposure again, giving ClaimAmount/Exposure² — a meaningless target.
tabpfn_holdout_configs = [
    {'model_factory': lambda: TabPFNWrapper(task='regression', n_train_max=10_000,
                                            exposure_strategy='feature', predict_batch_size=5_000),
     'pipeline_factory': RawFeaturePipeline,
     'approach': 'tweedie', 'model_name': 'TabPFN_10K', 'features_label': 'raw'},
    {'model_factory': lambda: TabPFNWrapper(task='regression', n_train_max=10_000,
                                            exposure_strategy='feature', predict_batch_size=5_000),
     'pipeline_factory': GBMFeaturePipeline,
     'approach': 'tweedie', 'model_name': 'TabPFN_10K', 'features_label': 'gbm'},
]

tabpfn_metrics = holdout_evaluation(
    model_configs=tabpfn_holdout_configs,
    X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
    X_holdout=X_holdout, y_holdout=y_holdout, exposure_holdout=exp_holdout,
    tag='tabpfn',
)
tabpfn_preds = pd.read_parquet(holdout_dir / 'predictions_tabpfn.parquet')
for col in tabpfn_preds.columns:
    all_preds[col] = tabpfn_preds[col].values
all_metrics.append(tabpfn_metrics)

# --- Hurdle: TabPFN cls + GammaGLM ---
if 'TabPFN_hurdle_cls_gamma' in tabpfn_cv:
    preds_h1, metrics_h1 = holdout_evaluation_hurdle(
        stage1_factory=lambda: TabPFNWrapper(task='classification', n_train_max=10_000, predict_batch_size=5_000),
        stage2_factory=GammaGLM,
        feature_pipeline_factory=GBMFeaturePipeline,
        model_name='TabPFN_hurdle_cls_gamma', features_label='gbm',
        X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
        X_holdout=X_holdout, y_holdout=y_holdout, exposure_holdout=exp_holdout,
    )
    all_preds['TabPFN_hurdle_cls_gamma_gbm_hurdle'] = preds_h1
    all_metrics.append(pd.DataFrame([metrics_h1]).set_index('model'))
    print(f"Hurdle cls+Gamma holdout: {metrics_h1}")

# --- Hurdle: TabPFN cls + TabPFN sev ---
if 'TabPFN_hurdle_full' in tabpfn_cv:
    preds_h2, metrics_h2 = holdout_evaluation_hurdle(
        stage1_factory=lambda: TabPFNWrapper(task='classification', n_train_max=10_000, predict_batch_size=5_000),
        stage2_factory=lambda: TabPFNWrapper(task='regression', n_train_max=10_000,
                                             exposure_strategy='feature', predict_batch_size=5_000),
        feature_pipeline_factory=GBMFeaturePipeline,
        model_name='TabPFN_hurdle_full', features_label='gbm',
        X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
        X_holdout=X_holdout, y_holdout=y_holdout, exposure_holdout=exp_holdout,
    )
    all_preds['TabPFN_hurdle_full_gbm_hurdle'] = preds_h2
    all_metrics.append(pd.DataFrame([metrics_h2]).set_index('model'))
    print(f"Hurdle full holdout: {metrics_h2}")

# --- Save TabPFN-only predictions (Tweedie + hurdle) so post2_analysis rebuild works ---
tabpfn_preds_df = pd.DataFrame(all_preds)
tabpfn_preds_df.to_parquet(holdout_dir / 'predictions_tabpfn.parquet')
print(f"TabPFN predictions (incl. hurdle) saved: {list(tabpfn_preds_df.columns)}")

# --- Merge TabPFN predictions with baseline predictions ---
baseline_preds_path = holdout_dir / 'predictions_baselines.parquet'
if baseline_preds_path.exists():
    baseline_preds = pd.read_parquet(baseline_preds_path)
    for col in baseline_preds.columns:
        all_preds[col] = baseline_preds[col].values
    print(f"Merged {len(baseline_preds.columns)} baseline prediction columns")
else:
    print("WARNING: predictions_baselines.parquet not found — run post2_baselines.ipynb first")

merged_preds_df = pd.DataFrame(all_preds)
merged_preds_df.to_parquet(holdout_dir / 'predictions_all.parquet')
print(f"\nMerged predictions saved: {list(merged_preds_df.columns)}")

merged_metrics = pd.concat(all_metrics)
merged_metrics.to_parquet(holdout_dir / 'metrics_tabpfn.parquet')
print(f"\nTabPFN metrics saved → results/holdout/metrics_tabpfn.parquet")
print("Copy results/ back to local before stopping pod.")

## Step 8 — Deployment Benchmarks: TabPFN

Measures inference latency, peak memory, and serialised model size for TabPFN configs.
Saves to `results/deployment/benchmarks_tabpfn.parquet` — merged with baseline benchmarks
in `post2_analysis.ipynb`.

In [ ]:
import time, tracemalloc, pickle

deploy_dir = Path(project_root) / 'results' / 'deployment'
deploy_dir.mkdir(parents=True, exist_ok=True)

# Prepare a fixed 10K holdout slice — raw features throughout
BENCH_N = 10_000
pipe_bench = RawFeaturePipeline()
X_bench_raw = X_holdout.head(BENCH_N)
X_bench = pipe_bench.fit_transform(X_bench_raw)
exp_bench = exp_holdout.head(BENCH_N).values

# Train on 50K dev subsample — raw features
TRAIN_N = 50_000
rng = np.random.default_rng(42)
train_idx = rng.choice(len(X_dev), size=TRAIN_N, replace=False)
X_fit_raw = X_dev.iloc[train_idx]
y_fit     = y_dev['pure_premium'].iloc[train_idx].values
exp_fit   = exp_dev.iloc[train_idx].values

pipe_fit = RawFeaturePipeline()
X_fit = pipe_fit.fit_transform(X_fit_raw)
sev_mask_fit = ~np.isnan(y_dev['severity'].iloc[train_idx].values)
X_fit_claims = X_fit[sev_mask_fit]
y_fit_claims = np.clip(y_fit[sev_mask_fit], 1e-10,
                       np.percentile(y_fit[sev_mask_fit], 99))
exp_fit_claims = exp_fit[sev_mask_fit]

def bench_model(name, model, X_pred, exposure=None):
    """Time 5 inference runs, measure peak GPU VRAM, estimate size."""
    times = []
    for _ in range(5):
        t0 = time.perf_counter()
        model.predict(X_pred, exposure=exposure) if exposure is not None else model.predict(X_pred)
        times.append(time.perf_counter() - t0)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        model.predict(X_pred, exposure=exposure) if exposure is not None else model.predict(X_pred)
        peak_mb = torch.cuda.max_memory_allocated() / 1e6
    else:
        tracemalloc.start()
        model.predict(X_pred, exposure=exposure) if exposure is not None else model.predict(X_pred)
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        peak_mb = peak / 1e6

    try:
        size_mb = len(pickle.dumps(model)) / 1e6
    except Exception:
        size_mb = float('nan')

    return {
        'model':            name,
        'median_latency_s': round(float(np.median(times)), 4),
        'peak_memory_mb':   round(peak_mb, 1),
        'model_size_mb':    round(size_mb, 2),
    }

rows = []

# --- TabPFN 10K raw ---
m_raw = TabPFNWrapper(task='regression', n_train_max=10_000,
                     exposure_strategy='feature', predict_batch_size=5_000)
m_raw.fit(X_fit, y_fit, exposure=exp_fit)
rows.append(bench_model('TabPFN_10K_raw', m_raw, X_bench, exposure=exp_bench))
print(f"TabPFN_10K_raw: {rows[-1]}")

# --- Hurdle: TabPFN cls + GammaGLM (raw features) ---
s1_cg = TabPFNWrapper(task='classification', n_train_max=10_000, predict_batch_size=5_000)
s1_cg.fit(X_fit, (y_fit > 0).astype(int), exposure=exp_fit)
s2_cg = GammaGLM()
s2_cg.fit(X_fit_claims, y_fit_claims, exposure=exp_fit_claims)

class _HurdleCG:
    def predict(self, X):
        p = np.clip(s1_cg.predict(X), 1e-6, 1 - 1e-6)
        s = s2_cg.predict(X, exposure=exp_bench)
        return p * s

rows.append(bench_model('TabPFN_hurdle_cls_gamma', _HurdleCG(), X_bench))
print(f"TabPFN_hurdle_cls_gamma: {rows[-1]}")

# --- Hurdle: TabPFN cls + TabPFN sev (raw features) ---
s1_ff = TabPFNWrapper(task='classification', n_train_max=10_000, predict_batch_size=5_000)
s1_ff.fit(X_fit, (y_fit > 0).astype(int), exposure=exp_fit)
s2_ff = TabPFNWrapper(task='regression', n_train_max=10_000,
                     exposure_strategy='feature', predict_batch_size=5_000)
s2_ff.fit(X_fit_claims, y_fit_claims, exposure=exp_fit_claims)

class _HurdleFF:
    def predict(self, X):
        p = np.clip(s1_ff.predict(X), 1e-6, 1 - 1e-6)
        s = s2_ff.predict(X, exposure=exp_bench)
        return p * s

rows.append(bench_model('TabPFN_hurdle_full', _HurdleFF(), X_bench))
print(f"TabPFN_hurdle_full: {rows[-1]}")

# --- Save ---
bench_df = pd.DataFrame(rows).set_index('model')
print("\n=== TabPFN Deployment Benchmarks ===")
print(bench_df)
bench_df.to_parquet(deploy_dir / 'benchmarks_tabpfn.parquet')
print(f"\nSaved → results/deployment/benchmarks_tabpfn.parquet")

## Step 9 — SHAP Interpretability

**Goal:** Compare what XGBoost and TabPFN have *learned* about insurance risk, not just how accurate they are.

**Protocol (from spec §9):**
- **TreeSHAP on XGBoost** — exact, fast (~seconds on 1,000 rows)
- **KernelSHAP on TabPFN** — approximate, slow (~minutes on 200 rows); 30-min timeout
- Feature importance rank correlation (Spearman ρ) between the two models
- Dependence plots for 4 key features: BonusMalus, DrivAge, VehPower, Density
- Waterfall plots for 5 individual policies

**Feature-space alignment:** both models are trained on the 15-column `GBMFeaturePipeline`.
For TabPFN we use `exposure_strategy='rate'` in this section — target = pure_premium / exposure,
so the model never sees exposure as a feature. This gives both models identical input spaces and
makes SHAP values directly comparable.

Results saved to `results/interpretability/` and `figures/post2/`.

In [ ]:
import shap
import scipy.stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

from src.models.gbm import XGBoostModel, LightGBMModel

interp_dir = Path(project_root) / 'results' / 'interpretability'
fig_dir    = Path(project_root) / 'figures' / 'post2'
interp_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# ── Shared subsample: 50K dev rows (same seed as deployment benchmarks) ──────
TRAIN_N_INTERP = 50_000
rng_interp = np.random.default_rng(42)
train_idx_interp = rng_interp.choice(len(X_dev), size=TRAIN_N_INTERP, replace=False)

X_dev_sub   = X_dev.iloc[train_idx_interp]
y_pp_sub    = y_dev['pure_premium'].iloc[train_idx_interp].values
exp_sub     = exp_dev.iloc[train_idx_interp].values

pipe_interp = GBMFeaturePipeline()
X_fit_interp = pipe_interp.fit_transform(X_dev_sub, pd.Series(y_pp_sub))
feat_names   = pipe_interp.feature_names_
print(f"Training features ({len(feat_names)}): {feat_names}")

# ── Holdout sample for explanation ───────────────────────────────────────────
# Spec: TreeSHAP on 1,000 rows; KernelSHAP on 200 rows.
N_EXPLAIN_TREE   = CONFIG.get('shap', {}).get('n_explain_treeshap',   1000)
N_EXPLAIN_KERNEL = CONFIG.get('shap', {}).get('n_explain_kernelshap',  200)
N_BACKGROUND     = CONFIG.get('shap', {}).get('n_background',          100)

pipe_holdout_interp = GBMFeaturePipeline()
pipe_holdout_interp._transformer = None
pipe_holdout_interp._target_encoders = pipe_interp._target_encoders
pipe_holdout_interp._fitted = True
pipe_holdout_interp.feature_names_ = feat_names

X_holdout_interp = pipe_interp.transform(X_holdout)
exp_holdout_arr  = exp_holdout.values

rng_h = np.random.default_rng(99)
explain_idx = rng_h.choice(len(X_holdout_interp), size=N_EXPLAIN_TREE, replace=False)
X_explain_tree   = X_holdout_interp[explain_idx]             # (1000, 15)
X_explain_kernel = X_explain_tree[:N_EXPLAIN_KERNEL]         # (200, 15)
exp_explain      = exp_holdout_arr[explain_idx]              # (1000,) for closure
exp_explain_k    = exp_explain[:N_EXPLAIN_KERNEL]            # (200,)

# Background dataset — sampled from training data with replacement=False
bg_idx  = rng_h.choice(len(X_fit_interp), size=N_BACKGROUND, replace=False)
X_bg    = X_fit_interp[bg_idx]

# ── Fit XGBoost (Tweedie, 1.5) ───────────────────────────────────────────────
print("\nFitting XGBoost (Tweedie) for interpretability…")
t0 = time.time()
xgb_interp = XGBoostModel(approach='tweedie', tweedie_power=1.5)
xgb_interp.fit(X_fit_interp, y_pp_sub, exposure=exp_sub)
print(f"  done in {time.time()-t0:.1f}s")

# ── Fit TabPFN (rate strategy — no exposure column) ──────────────────────────
print("Fitting TabPFN (rate strategy, 10K) for interpretability…")
t0 = time.time()
tabpfn_interp = TabPFNWrapper(
    task='regression',
    n_train_max=10_000,
    exposure_strategy='rate',    # target = PP/exposure; model sees 15 features only
    predict_batch_size=5_000,
)
tabpfn_interp.fit(X_fit_interp, y_pp_sub, exposure=exp_sub)
print(f"  done in {time.time()-t0:.1f}s")

print("\nModels ready. Background and explain sets prepared.")

In [ ]:
# ── TreeSHAP — XGBoost ───────────────────────────────────────────────────────
# TreeSHAP is exact for tree ensembles: O(TLD²) where T=trees, L=leaves, D=depth.
# No approximation needed; 1,000 rows takes seconds.

print("=== TreeSHAP: XGBoost ===")
t0 = time.time()
explainer_xgb = shap.TreeExplainer(xgb_interp._model)
shap_values_xgb = explainer_xgb.shap_values(X_explain_tree)   # (1000, 15)
t_treeshap = time.time() - t0
print(f"TreeSHAP (1,000 rows): {t_treeshap:.2f}s")
print(f"SHAP array shape: {shap_values_xgb.shape}")

# Sanity check: sum(SHAP) + expected_value ≈ model prediction for a few rows
preds_xgb_check = xgb_interp.predict(X_explain_tree[:5])
shap_sum_check  = shap_values_xgb[:5].sum(axis=1) + explainer_xgb.expected_value
print(f"\nSanity check (row 0): pred={preds_xgb_check[0]:.4f}, "
      f"SHAP sum + E[f(X)]={shap_sum_check[0]:.4f}")

# Global importance: mean |SHAP| per feature
importance_xgb = np.abs(shap_values_xgb).mean(axis=0)
feat_imp_xgb   = pd.Series(importance_xgb, index=feat_names).sort_values(ascending=False)
print("\nTop features (mean |SHAP|):")
print(feat_imp_xgb.head(10).round(4).to_string())

np.save(interp_dir / 'shap_values_xgb.npy', shap_values_xgb)
print(f"\nSaved → results/interpretability/shap_values_xgb.npy")

In [ ]:
# ── KernelSHAP — TabPFN ──────────────────────────────────────────────────────
# KernelSHAP is model-agnostic but approximate. It estimates SHAP values by
# fitting a weighted linear model over random feature coalitions.
#
# Cost: O(n_background × n_explain × 2^n_features coalitions sampled).
# With 100 background × 200 explain × 15 features → expect 5–30 min on GPU.
#
# Predict function: TabPFN (rate strategy) takes only the 15-column X.
# Exposure is handled internally (target = PP/exposure at train time;
# at predict time the wrapper multiplies back by exposure). For SHAP,
# we need a predict callable that accepts arbitrary-shaped X arrays.
# We pass a fixed mean exposure so SHAP attributions are for a "unit-exposure"
# policy — the shape of the risk curve rather than the absolute premium.
#
# Timeout guard: if >30 min, reduce N_EXPLAIN_KERNEL and continue.

TIMEOUT_MINUTES = CONFIG.get('shap', {}).get('timeout_minutes', 30)
MEAN_EXPOSURE   = float(np.mean(exp_sub))   # ~0.5 for freMTPL2

def tabpfn_predict_fn(X_shap: np.ndarray) -> np.ndarray:
    """Predict using TabPFN (rate strategy) with fixed unit exposure = mean."""
    n = len(X_shap)
    exp_fixed = np.full(n, MEAN_EXPOSURE, dtype=np.float32)
    return tabpfn_interp.predict(X_shap.astype(np.float32), exposure=exp_fixed)

print("=== KernelSHAP: TabPFN ===")
print(f"Background: {N_BACKGROUND} rows | Explain: {N_EXPLAIN_KERNEL} rows")
print(f"Timeout: {TIMEOUT_MINUTES} min | Mean exposure (fixed): {MEAN_EXPOSURE:.4f}")

explainer_tabpfn = shap.KernelExplainer(tabpfn_predict_fn, X_bg)

t0 = time.time()
shap_values_tabpfn = None
n_explain_actual   = N_EXPLAIN_KERNEL

try:
    import signal

    class _TimeoutError(Exception):
        pass

    def _timeout_handler(signum, frame):
        raise _TimeoutError

    signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(TIMEOUT_MINUTES * 60)

    shap_values_tabpfn = explainer_tabpfn.shap_values(
        X_explain_kernel, nsamples='auto', silent=False
    )
    signal.alarm(0)   # cancel alarm on success

except _TimeoutError:
    elapsed = time.time() - t0
    # Reduce to 50 rows and try again — document the timeout
    n_explain_actual = 50
    print(f"\nWARNING: KernelSHAP timeout after {elapsed/60:.1f} min. "
          f"Retrying with {n_explain_actual} rows.")
    shap_values_tabpfn = explainer_tabpfn.shap_values(
        X_explain_kernel[:n_explain_actual], nsamples='auto', silent=False
    )

except AttributeError:
    # SIGALRM not available on Windows — run without timeout
    shap_values_tabpfn = explainer_tabpfn.shap_values(
        X_explain_kernel, nsamples='auto', silent=False
    )

t_kernelshap = time.time() - t0
speedup_ratio = t_kernelshap / max(t_treeshap, 1e-6)

print(f"\nKernelSHAP ({n_explain_actual} rows): {t_kernelshap:.1f}s  "
      f"({t_kernelshap/60:.1f} min)")
print(f"Slowdown vs TreeSHAP: {speedup_ratio:.0f}×")
print(f"SHAP array shape: {np.array(shap_values_tabpfn).shape}")

# Global importance
importance_tabpfn = np.abs(shap_values_tabpfn).mean(axis=0)
feat_imp_tabpfn   = pd.Series(importance_tabpfn, index=feat_names).sort_values(ascending=False)
print("\nTop features (mean |SHAP|):")
print(feat_imp_tabpfn.head(10).round(6).to_string())

np.save(interp_dir / 'shap_values_tabpfn.npy', shap_values_tabpfn)
print(f"\nSaved → results/interpretability/shap_values_tabpfn.npy")

In [ ]:
# ── Feature Importance: Rank Correlation ─────────────────────────────────────
# We compare mean |SHAP| rankings between XGBoost and TabPFN.
# Both cover all 15 GBMFeaturePipeline features.
#
# For TabPFN, SHAP was computed on N_EXPLAIN_KERNEL rows (possibly reduced by
# timeout). XGBoost used 1,000 rows. We compare mean |SHAP| ranks.
#
# A high Spearman ρ means both models agreed on which features matter most —
# suggesting TabPFN is learning similar signals to a tuned Tweedie GBM.
# A low ρ is an interesting finding: the models may have found different
# risk signals, or KernelSHAP's approximation is noisy.

n_kernel_rows = len(shap_values_tabpfn)
imp_xgb_on_kernel = np.abs(shap_values_xgb[:n_kernel_rows]).mean(axis=0)

rho, pval = scipy.stats.spearmanr(imp_xgb_on_kernel, importance_tabpfn)
print(f"Feature importance rank correlation (Spearman ρ): {rho:.4f}  (p={pval:.4f})")
print(f"  — computed on the {n_kernel_rows}-row overlap between both SHAP runs\n")

# Side-by-side importance table
imp_df = pd.DataFrame({
    'XGBoost_mean_abs_shap':  pd.Series(imp_xgb_on_kernel, index=feat_names),
    'TabPFN_mean_abs_shap':   pd.Series(importance_tabpfn,  index=feat_names),
}).assign(
    xgb_rank   = lambda d: d['XGBoost_mean_abs_shap'].rank(ascending=False).astype(int),
    tabpfn_rank = lambda d: d['TabPFN_mean_abs_shap'].rank(ascending=False).astype(int),
    rank_delta  = lambda d: (d['xgb_rank'] - d['tabpfn_rank']).abs(),
).sort_values('xgb_rank')

print("Feature importance comparison (XGBoost rank order):")
print(imp_df[['xgb_rank', 'tabpfn_rank', 'rank_delta',
              'XGBoost_mean_abs_shap', 'TabPFN_mean_abs_shap']].round(6).to_string())

# Save
imp_df.to_parquet(interp_dir / 'feature_importance_comparison.parquet')
import json
agreement = {
    'spearman_rho':  round(float(rho),  4),
    'spearman_pval': round(float(pval), 4),
    'n_rows_kernel': int(n_kernel_rows),
    'n_rows_tree':   int(N_EXPLAIN_TREE),
    't_treeshap_s':  round(float(t_treeshap),    2),
    't_kernelshap_s': round(float(t_kernelshap), 1),
    'speedup_ratio': round(float(speedup_ratio), 0),
}
with open(interp_dir / 'agreement.json', 'w') as f:
    json.dump(agreement, f, indent=2)
print(f"\nSaved → results/interpretability/agreement.json")

In [ ]:
# ── Dependence Plots: 4 Key Features ─────────────────────────────────────────
# For each feature: SHAP value (y) vs raw feature value (x), one panel per model.
# Side-by-side layout: XGBoost (left) | TabPFN (right).
#
# Key features from spec §9.2:
#   BonusMalus — primary pricing signal, non-linear
#   DrivAge    — U-shaped actuarial age curve
#   VehPower   — flattens above ~10
#   Density    — urban/rural gradient (log-transformed in GBMFeaturePipeline)
#
# For BonusMalus and DrivAge we use the raw feature values from X_holdout
# (available in the original DataFrame) for interpretable x-axes.
# For log_density we label the x-axis "log1p(Density)".

KEY_FEATURES = [
    ('BonusMalus',  'BonusMalus',  'BonusMalus'),
    ('DrivAge',     'DrivAge',     'DrivAge'),
    ('VehPower',    'VehPower',    'VehPower'),
    ('log_density', 'log_density', 'log1p(Density)'),
]

# Raw values from the holdout subset we explained (for x-axis labels)
X_holdout_sub_df = X_holdout.iloc[explain_idx[:n_kernel_rows]].reset_index(drop=True)

# Align raw BonusMalus etc. with the kernel explain subset
raw_vals = {
    'BonusMalus':  X_holdout_sub_df['BonusMalus'].values,
    'DrivAge':     X_holdout_sub_df['DrivAge'].values,
    'VehPower':    X_holdout_sub_df['VehPower'].values,
    'log_density': X_explain_kernel[:n_kernel_rows, feat_names.index('log_density')],
}

for feat_key, raw_key, x_label in KEY_FEATURES:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
    fig.suptitle(f'SHAP Dependence: {x_label}', fontsize=13, fontweight='bold')

    feat_idx  = feat_names.index(feat_key)
    x_vals    = raw_vals[raw_key]
    shap_xgb  = shap_values_xgb[:n_kernel_rows, feat_idx]
    shap_tpfn = shap_values_tabpfn[:, feat_idx]

    for ax, shap_v, title, color in [
        (axes[0], shap_xgb,  'XGBoost (TreeSHAP)',  '#2196F3'),
        (axes[1], shap_tpfn, 'TabPFN (KernelSHAP)', '#FF6F00'),
    ]:
        ax.scatter(x_vals, shap_v, alpha=0.35, s=12, c=color, rasterized=True)
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
        ax.set_xlabel(x_label, fontsize=11)
        ax.set_ylabel('SHAP value', fontsize=11)
        ax.set_title(title, fontsize=11)
        # Smoothed trend line
        sort_idx = np.argsort(x_vals)
        from scipy.ndimage import uniform_filter1d
        smoothed = uniform_filter1d(shap_v[sort_idx], size=max(1, len(sort_idx)//20))
        ax.plot(x_vals[sort_idx], smoothed, color='black', linewidth=1.5, label='trend')

    plt.tight_layout()
    fname = fig_dir / f'shap_comparison_{feat_key}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → {fname}")

In [ ]:
# ── Waterfall Plots: 5 Individual Policies ───────────────────────────────────
# We show SHAP waterfalls for 5 representative policies from the kernel-explain
# subset (so both models have SHAP values for the same rows).
#
# Policy selection criteria (spec §9.3):
#   A — lowest predicted risk by XGBoost        (safest driver)
#   B — median predicted risk by XGBoost
#   C — highest predicted risk by XGBoost       (most dangerous driver)
#   D — largest absolute disagreement between XGBoost and TabPFN predictions
#   E — an outlier: TabPFN ranked much higher risk than XGBoost
#
# For each policy we show XGBoost waterfall (left) and TabPFN waterfall (right).
# SHAP waterfall plots require shap>=0.40; we fall back to bar charts on older
# versions.

preds_xgb_k   = xgb_interp.predict(X_explain_kernel[:n_kernel_rows])
exp_k_fixed   = np.full(n_kernel_rows, MEAN_EXPOSURE, dtype=np.float32)
preds_tabpfn_k = tabpfn_interp.predict(
    X_explain_kernel[:n_kernel_rows].astype(np.float32), exposure=exp_k_fixed
)

policy_labels = {
    'low_risk':    int(np.argmin(preds_xgb_k)),
    'median_risk': int(np.argsort(preds_xgb_k)[len(preds_xgb_k)//2]),
    'high_risk':   int(np.argmax(preds_xgb_k)),
    'max_disagree_abs': int(np.argmax(np.abs(preds_xgb_k - preds_tabpfn_k))),
    'tabpfn_higher': int(np.argmax(preds_tabpfn_k - preds_xgb_k)),
}
print("Selected policies (row indices within kernel-explain subset):")
for label, idx in policy_labels.items():
    print(f"  {label:20s} row={idx:3d}  "
          f"xgb={preds_xgb_k[idx]:.4f}  tabpfn={preds_tabpfn_k[idx]:.4f}")

def _waterfall_fallback(ax, shap_vals, feat_names, base_val, title):
    """Horizontal bar chart as waterfall fallback when shap.plots unavailable."""
    order = np.argsort(np.abs(shap_vals))[::-1][:10]
    colors = ['#e74c3c' if v > 0 else '#3498db' for v in shap_vals[order]]
    ax.barh(
        [feat_names[i] for i in order],
        shap_vals[order],
        color=colors,
    )
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('SHAP value', fontsize=9)
    ax.set_title(title, fontsize=9)

for label, row_idx in policy_labels.items():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f'Policy: {label.replace("_", " ")}  |  '
        f'XGB={preds_xgb_k[row_idx]:.4f}  TabPFN={preds_tabpfn_k[row_idx]:.4f}',
        fontsize=11, fontweight='bold'
    )

    sv_xgb   = shap_values_xgb[row_idx]
    sv_tpfn  = shap_values_tabpfn[row_idx]
    base_xgb = float(explainer_xgb.expected_value)
    base_tpfn = float(explainer_tabpfn.expected_value)

    try:
        # shap >= 0.40: use Explanation objects for proper waterfall
        expl_xgb = shap.Explanation(
            values=sv_xgb,
            base_values=base_xgb,
            data=X_explain_kernel[row_idx],
            feature_names=feat_names,
        )
        expl_tpfn = shap.Explanation(
            values=sv_tpfn,
            base_values=base_tpfn,
            data=X_explain_kernel[row_idx],
            feature_names=feat_names,
        )
        plt.sca(axes[0])
        shap.plots.waterfall(expl_xgb,  max_display=10, show=False)
        axes[0].set_title('XGBoost (TreeSHAP)', fontsize=10)
        plt.sca(axes[1])
        shap.plots.waterfall(expl_tpfn, max_display=10, show=False)
        axes[1].set_title('TabPFN (KernelSHAP)', fontsize=10)
    except Exception:
        _waterfall_fallback(axes[0], sv_xgb,  feat_names, base_xgb,  'XGBoost (TreeSHAP)')
        _waterfall_fallback(axes[1], sv_tpfn,  feat_names, base_tpfn, 'TabPFN (KernelSHAP)')

    plt.tight_layout()
    fname = fig_dir / f'waterfall_policy_{label}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → {fname}")

In [ ]:
# ── Global Summary: SHAP Bar Chart ───────────────────────────────────────────
# A single figure comparing mean |SHAP| across all 15 features for both models.
# This is the headline interpretability chart for the blog post.

fig, ax = plt.subplots(figsize=(9, 6))

n_feat = len(feat_names)
y_pos  = np.arange(n_feat)
width  = 0.38

# Normalise to [0,1] so both models are on the same scale
norm_xgb   = imp_xgb_on_kernel  / (imp_xgb_on_kernel.max()  + 1e-12)
norm_tpfn  = importance_tabpfn  / (importance_tabpfn.max()   + 1e-12)

# Sort by XGBoost importance
order = np.argsort(norm_xgb)
ax.barh(y_pos - width/2, norm_xgb[order],  width, label='XGBoost (TreeSHAP)',   color='#2196F3', alpha=0.85)
ax.barh(y_pos + width/2, norm_tpfn[order], width, label='TabPFN (KernelSHAP)', color='#FF6F00', alpha=0.85)

ax.set_yticks(y_pos)
ax.set_yticklabels([feat_names[i] for i in order], fontsize=9)
ax.set_xlabel('Normalised mean |SHAP|  (1.0 = most important feature for that model)', fontsize=9)
ax.set_title(
    f'Feature Importance: XGBoost vs TabPFN\n'
    f'Spearman ρ = {rho:.2f}  |  XGB: {N_EXPLAIN_TREE} rows  |  TabPFN: {n_kernel_rows} rows',
    fontsize=10,
)
ax.legend(fontsize=9)
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
fname = fig_dir / 'shap_importance_comparison.png'
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {fname}")

# ── Print interpretability summary ───────────────────────────────────────────
print("\n=== Interpretability Summary ===")
print(f"TreeSHAP   (XGBoost, {N_EXPLAIN_TREE} rows):  {t_treeshap:.2f}s")
print(f"KernelSHAP (TabPFN,  {n_kernel_rows} rows):  {t_kernelshap:.1f}s  ({t_kernelshap/60:.1f} min)")
print(f"Speed ratio:  {speedup_ratio:.0f}× slower (KernelSHAP vs TreeSHAP)")
print(f"Rank correlation (Spearman ρ): {rho:.4f}  (p={pval:.4f})")
print(f"\nArtifacts:")
print(f"  results/interpretability/shap_values_xgb.npy")
print(f"  results/interpretability/shap_values_tabpfn.npy")
print(f"  results/interpretability/feature_importance_comparison.parquet")
print(f"  results/interpretability/agreement.json")
print(f"  figures/post2/shap_comparison_{{feature}}.png  (×4)")
print(f"  figures/post2/waterfall_policy_{{label}}.png   (×5)")
print(f"  figures/post2/shap_importance_comparison.png")
print("\nCopy results/ and figures/ back to local before stopping pod.")